# Notebook 05 — Evaluation & Comparative Analysis

This notebook performs the complete evaluation of the reliability-aware RAG system built in Notebooks 02–04 and compares it against the baseline system from Semester 2.

## Structure

| Section | What it does |
|---------|-------------|
| **0** | Installation and package setup |
| **1** | Google Drive mount and path configuration |
| **2** | Load baseline system and pre-built reliable results |
| **3** | IR retrieval quality metrics (Baseline vs. Reliable) |
| **4** | Answer quality metrics: BLEU, ROUGE, Semantic Similarity |
| **5** | Reliability-oriented behavior metrics |
| **6** | Ablation study: contribution of individual mechanisms |
| **7** | Extended benchmark evaluation (challenging cases) |
| **8** | Qualitative examples |
| **9** | Visualizations and saving all results |

All outputs are saved to Google Drive (`Adv_GenAI_FS26/results/`) so they survive session resets and can be loaded directly into the Quarto report.

## Prerequisites

- Notebook 04 has been run and `reliable_pipeline_results.csv` exists in Drive
- Notebooks 02–03 have been run: BM25, Dense, and GraphRAG indices are built in Drive

## Section 0 — Installation

We clone the repository and install the `advanced-genai-rag` package together with
evaluation-specific dependencies that are not part of the core package:

- `rouge-score` — Google's ROUGE implementation
- `nltk` — provides BLEU scoring
- `scipy` — Spearman correlation for trust calibration
- `pytrec-eval-terrier` — IR evaluation (P@k, MRR, NDCG)

The `sentence-transformers` and `faiss-cpu` packages are re-installed explicitly
because Colab's pre-installed versions sometimes conflict with the package requirements.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("ADV_GENAI_RAG_REPO", "https://github.com/dhub100/advanced-genai-rag.git")
REPO_REF = os.environ.get("ADV_GENAI_RAG_REF", "main")
REPO_DIR = pathlib.Path("/content/advanced-genai-rag")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)

# Core package
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir",
     "sentence-transformers", "faiss-cpu"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir",
     str(REPO_DIR / "package")],
    check=True,
)

# Evaluation-specific dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "rouge-score", "nltk", "scipy", "pytrec-eval-terrier"],
    check=True,
)

# spaCy models (needed if groundedness verifier is re-used)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
subprocess.run([sys.executable, "-m", "spacy", "download", "de_core_news_sm"], check=False)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Reload package modules if already imported
for name in list(sys.modules):
    if name == "rag" or name.startswith("rag."):
        del sys.modules[name]

sys.path.insert(0, str(REPO_DIR / "package" / "src"))
sys.path.insert(0, str(REPO_DIR / "package"))

import rag
print(f"Package loaded from: {rag.__file__}")
print("Setup complete.")

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

## Section 1 — Google Drive & Path Configuration

All evaluation outputs are written to `Adv_GenAI_FS26/results/` on Drive.  This
guarantees that results survive a Colab session reset.  The figures sub-folder
holds PNG charts that are referenced directly by the Quarto report.

Adjust `ROOT` if your Drive layout differs from the default.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
ROOT    = pathlib.Path("/content/drive/MyDrive/Adv_GenAI_FS26")
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"
PATH_QA           = REPO_DIR / "baseline/benchmark/benchmark_qa_bilingual.json"
PATH_QA_EXT       = REPO_DIR / "package/benchmark/benchmark_qa_extended.json"
PATH_QRELS        = REPO_DIR / "baseline/benchmark/score/fixed_size"
PATH_RELIABLE_CSV = RESULTS / "reliable_pipeline_results.csv"

for label, p in [
    ("BM25 pickle",        PATH_BM25_PICKLE),
    ("Dense loader",       PATH_DENSE_LOADER),
    ("GraphRAG loader",    PATH_GRAG_LOADER),
    ("QA benchmark",       PATH_QA),
    ("QA extended",        PATH_QA_EXT),
    ("Qrels folder",       PATH_QRELS),
    ("Reliable CSV",       PATH_RELIABLE_CSV),
]:
    ok = pathlib.Path(p).exists()
    print(f"  {'OK' if ok else 'MISSING':>8}  {label}: {p}")

## Section 2 — Load Baseline System & Reliable Results

### Why load both?

The comparison requires running the **baseline** orchestrator freshly (to generate
baseline answers for ROUGE/BLEU scoring) while loading the **reliable** pipeline
results from the CSV produced by Notebook 04.  Re-running the reliable pipeline
from scratch would cost additional API/GPU time; the saved CSV is sufficient for
all metrics that do not require re-running inference.

If `reliable_pipeline_results.csv` is missing (first run), the reliable pipeline
is executed here and the CSV is saved before proceeding.

In [ ]:
import __main__
import torch

if not hasattr(torch, "is_available"):
    torch.is_available = torch.cuda.is_available

from rag.retrieval.agents.bm25 import BilingualBM25, QEBM25, load_bm25_fixed_qe
from rag.retrieval.translator import EnDeTranslator
__main__.BilingualBM25 = BilingualBM25
__main__.QEBM25 = QEBM25
__main__.EnDeTranslator = EnDeTranslator

from rag.retrieval.agents.dense import DenseRetriever, load_dense_fixed
from rag.retrieval.agents.graph import GraphAgent, load_graph_rag
from rag.retrieval.agents.answer_synthesizer import AnswerSynthesizerAgent
from rag.retrieval.orchestrator import Orchestrator

print("Loading BM25...")
bm25_fixed_qe = load_bm25_fixed_qe(bm25_pickle_path=str(PATH_BM25_PICKLE))
print("Loading Dense retriever...")
dense_fixed   = load_dense_fixed(dense_loader_path=str(PATH_DENSE_LOADER))
print("Loading GraphRAG...")
graph_rag     = load_graph_rag(loader_path=str(PATH_GRAG_LOADER))
print("Loading AnswerSynthesizer (Mistral-7B)...")
synthesizer   = AnswerSynthesizerAgent()

baseline_orchestrator = Orchestrator(
    bm25=bm25_fixed_qe, dense=dense_fixed,
    graph=graph_rag, synthesizer=synthesizer
)
print("Baseline system ready.")

In [ ]:
from rag.reliability.evidence_sufficiency import EvidenceSufficiencyChecker
from rag.reliability.groundedness import GroundednessVerifier
from rag.reliability.trust_scorer import TrustScorer
from rag.reliability.reliable_orchestrator import ReliableOrchestrator, ReliableResponse

def embed_fn(texts: list) -> np.ndarray:
    return np.array(dense_fixed.store._embedding_function.embed_documents(texts))

groundedness_verifier = GroundednessVerifier()
trust_scorer = TrustScorer()

try:
    from google.colab import userdata
    openai_api_key = userdata.get("OPENAI_API_KEY")
    if openai_api_key:
        from openai import OpenAI
        openai_client = OpenAI(api_key=openai_api_key)
        print("OpenAI client ready — LLM-based query rewriting enabled.")
    else:
        openai_client = None
        print("OPENAI_API_KEY not set — rule-based query rewrite fallback.")
except Exception:
    openai_client = None

reliable_orchestrator = ReliableOrchestrator(
    orchestrator=baseline_orchestrator,
    embed_fn=embed_fn,
    max_retries=2,
    openai_client=openai_client,
    groundedness_verifier=groundedness_verifier,
    trust_scorer=trust_scorer,
)
print("ReliableOrchestrator ready.")

In [ ]:
# Load benchmark
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

with open(PATH_QA_EXT, "r", encoding="utf-8") as f:
    qa_extended = json.load(f)

print(f"Standard benchmark: {len(qa_data)} questions")
print(f"Extended benchmark: {len(qa_extended)} challenging questions")

In [ ]:
# Load reliable results from Drive CSV (produced by Notebook 04, Section 10)
# If the CSV does not exist yet, run the reliable pipeline and save it.
if PATH_RELIABLE_CSV.exists():
    rel_df = pd.read_csv(PATH_RELIABLE_CSV)
    reliable_results = rel_df.to_dict(orient="records")
    print(f"Loaded {len(reliable_results)} reliable results from Drive.")
else:
    print("reliable_pipeline_results.csv not found — running reliable pipeline now...")
    reliable_results = []
    questions, gold_answers = [], []
    for x in qa_data:
        questions.append(x["question"])
        gold_answers.append(x["answer"])
        questions.append(x["question_de"])
        gold_answers.append(x["answer_de"])

    for q, a in tqdm(zip(questions, gold_answers), total=len(questions), desc="Reliable pipeline"):
        t0 = time.time()
        r  = reliable_orchestrator.run(query=q, strategy="confidence", top_k=5)
        reliable_results.append({
            "query":             q,
            "gold":              a,
            "answer":            r.answer,
            "abstained":         r.abstained,
            "trigger":           r.abstention.trigger if r.abstained and r.abstention else None,
            "recoveries":        r.recovery_attempts,
            "ev_score":          r.evidence_report.score if r.evidence_report else None,
            "groundedness_score": r.groundedness_score,
            "trust_score":       r.trust_score,
            "latency":           time.time() - t0,
        })

    rel_df = pd.DataFrame(reliable_results)
    rel_df.to_csv(PATH_RELIABLE_CSV, index=False)
    print(f"Saved {len(reliable_results)} results to Drive.")

## Section 3 — IR Retrieval Quality: Baseline vs. Reliable

### Why IR metrics?

Before comparing answer quality we evaluate **retrieval quality** independently.
The reliable orchestrator wraps the same underlying retrieval agents as the baseline;
the reliability layer (mechanisms A, G, E) is a *post-retrieval* decision policy.
Comparing IR metrics tells us whether reliability mechanisms accidentally degrade
the retrieved evidence set — e.g. by switching strategies during recovery.

### Metrics used

| Metric | What it measures |
|--------|------------------|
| **P@k** (k=1,3,5,10) | Precision: fraction of top-k results that are relevant |
| **Recall@k** (k=5,10,100) | Fraction of all relevant documents retrieved in top-k |
| **MRR** | Mean Reciprocal Rank — how high the first relevant result appears |
| **NDCG@k** (k=5,10) | Normalized Discounted Cumulative Gain — graded relevance |

Relevance judgements come from `baseline/benchmark/score/fixed_size/` — GPT-4o-mini
scored each retrieved chunk on a 0–1 scale and those scores are the ground truth.

### Design note on abstention

The reliable orchestrator *always* retrieves documents, even for queries it later
abstains on.  We use the retrieved documents (not the final answer) for IR scoring.
This is the correct comparison: both systems see the same corpus; only the
post-retrieval decision differs.

In [ ]:
from rag.evaluation.metrics import load_qrels, METRICS
from rag.evaluation.evaluator import ComprehensiveEvaluator

qrels = load_qrels(str(PATH_QRELS))
print(f"Loaded qrels for {len(qrels)} queries.")

evaluator = ComprehensiveEvaluator(qrels)

In [ ]:
def make_retriever_fn(orchestrator, strategy):
    """Wrap an orchestrator so ComprehensiveEvaluator can call it.

    ComprehensiveEvaluator expects a callable (query, top_k) -> (docs, trace).
    ReliableOrchestrator.run() returns a ReliableResponse dataclass, so we
    unpack it here.  The baseline orchestrator returns a plain dict.
    """
    def fn(query, top_k=100):
        result = orchestrator.run(strategy=strategy, query=query, top_k=top_k)
        if isinstance(result, dict):
            docs = result.get("documents", [])
        else:
            docs = result.documents
        return docs, {}
    return fn

# Evaluate baseline (confidence strategy — best single-strategy baseline)
evaluator.evaluate_retriever(
    make_retriever_fn(baseline_orchestrator, "confidence"),
    qa_data, name="Baseline-Confidence"
)

# Evaluate reliable pipeline (same confidence strategy, wrapped with reliability)
evaluator.evaluate_retriever(
    make_retriever_fn(reliable_orchestrator, "confidence"),
    qa_data, name="Reliable-Confidence"
)

In [ ]:
ir_comparison = evaluator.compare_strategies()
print(ir_comparison.to_string(index=False))

# Save for report
ir_comparison.to_csv(RESULTS / "ir_metrics_comparison.csv", index=False)
print("Saved: ir_metrics_comparison.csv")

In [ ]:
# Statistical significance test on MRR (most commonly reported IR metric)
sig = evaluator.statistical_significance_test(
    "Baseline-Confidence", "Reliable-Confidence", metric="recip_rank"
)
print(f"Paired t-test on MRR:")
print(f"  t-statistic : {sig['t_statistic']:.4f}")
print(f"  p-value     : {sig['p_value']:.4f}")
print(f"  significant : {sig['significant']}  (threshold p < 0.05)")
print(f"  mean diff   : {sig['mean_diff']:+.4f}  (Reliable − Baseline)")

import json as _json
with open(RESULTS / "stat_significance.json", "w") as f:
    _json.dump(sig, f, indent=2)
print("Saved: stat_significance.json")

## Section 4 — Answer Quality: BLEU, ROUGE, Semantic Similarity

### Why three metrics?

Answer quality is multi-dimensional.  We use three complementary metrics:

- **BLEU** — n-gram precision (emphasis on exact matches); sensitive to
  word choice and useful as a complementary signal to ROUGE.
- **ROUGE-1 / ROUGE-L** — recall-oriented; captures whether the main content
  of the gold answer appears in the generated answer.
- **Semantic similarity** — cosine of multilingual-E5 embeddings; robust to
  paraphrase and captures meaning even when surface wording differs.

### Three-way split: why a direct comparison is unfair

The reliable system abstains on a fraction of queries where it judges the
evidence insufficient.  The baseline *always* answers (even when it hallucinates).
A naive comparison that scores abstained queries as ROUGE=0 would penalise
correct abstention.  We therefore split the queries into:

- **Subset A** — both systems answered.  Fair head-to-head comparison.
- **Subset B** — reliable abstained, baseline answered.  We show the baseline
  ROUGE on these queries: if it is low, the reliable system correctly withheld a
  hallucinated or poor-quality answer.

This framing captures the central argument of the reliability approach:
**correct abstention is better than a wrong answer**.

In [ ]:
# Generate baseline answers for all standard benchmark queries
print("Generating baseline answers...")
baseline_answers: dict[str, dict] = {}
for q in tqdm(qa_data, desc="Baseline answers"):
    for lang in ("question", "question_de"):
        ans_key = "answer" if lang == "question" else "answer_de"
        query   = q[lang]
        t0      = time.time()
        result  = baseline_orchestrator.run(strategy="confidence", query=query, top_k=5)
        baseline_answers[query] = {
            "answer": result["answer"],
            "gold":   q[ans_key],
            "latency": time.time() - t0,
        }

baseline_df = pd.DataFrame.from_dict(baseline_answers, orient="index").reset_index()
baseline_df.columns = ["query", "baseline_answer", "gold", "latency"]
baseline_df.to_csv(RESULTS / "baseline_answers.csv", index=False)
print(f"Saved {len(baseline_df)} baseline answers.")

In [ ]:
# Build Subset A (both answered) and Subset B (reliable abstained)
subset_a, subset_b = [], []

for rec in reliable_results:
    q = rec["query"]
    if q not in baseline_answers:
        continue
    bl = baseline_answers[q]
    if not rec["abstained"]:
        subset_a.append({
            "query":           q,
            "baseline_answer": bl["answer"],
            "reliable_answer": rec["answer"],
            "gold":            bl["gold"],
        })
    else:
        subset_b.append({
            "query":           q,
            "baseline_answer": bl["answer"],
            "gold":            bl["gold"],
            "trigger":         rec["trigger"],
        })

print(f"Subset A (both answered)  : {len(subset_a)} queries")
print(f"Subset B (reliable abstained, baseline answered): {len(subset_b)} queries")

In [ ]:
# Evaluate answer quality per subset using ComprehensiveEvaluator
quality_a_baseline = ComprehensiveEvaluator.evaluate_answer_quality(
    [{"answer": r["baseline_answer"], "gold": r["gold"]} for r in subset_a],
    embed_fn=embed_fn,
)
quality_a_reliable = ComprehensiveEvaluator.evaluate_answer_quality(
    [{"answer": r["reliable_answer"], "gold": r["gold"]} for r in subset_a],
    embed_fn=embed_fn,
)
quality_b_baseline = ComprehensiveEvaluator.evaluate_answer_quality(
    [{"answer": r["baseline_answer"], "gold": r["gold"]} for r in subset_b],
    embed_fn=embed_fn,
)

answer_quality_df = pd.DataFrame(
    [quality_a_baseline, quality_a_reliable, quality_b_baseline],
    index=["Baseline (Subset A)", "Reliable (Subset A)", "Baseline on abstained queries (Subset B)"],
)
print(answer_quality_df.to_string())
answer_quality_df.to_csv(RESULTS / "answer_quality_comparison.csv")
print("Saved: answer_quality_comparison.csv")
print()
print("Interpretation:")
print(f"  Subset A   — {len(subset_a)} queries both systems answered.")
print(f"  Subset B   — {len(subset_b)} queries reliable abstained on.")
print(f"  Baseline ROUGE-L on Subset B: {quality_b_baseline['rougeL']:.3f}")
print("  If Subset B baseline ROUGE is low, abstention was the right call.")

In [ ]:
# Add per-query ROUGE-L to reliable_results for trust calibration
from rouge_score import rouge_scorer as _rs
_scorer = _rs.RougeScorer(["rougeL"], use_stemmer=False)

query_to_rouge: dict[str, float] = {}
for item in subset_a:
    r = _scorer.score(item["gold"].lower(), item["reliable_answer"].lower())
    query_to_rouge[item["query"]] = r["rougeL"].fmeasure

for rec in reliable_results:
    rec["rouge_l"] = query_to_rouge.get(rec["query"], None)

## Section 5 — Reliability-Oriented Behavior Metrics

Standard IR and ROUGE metrics evaluate *quality when the system answers*.  They
do not measure how well the system handles uncertainty.  This section computes
a complementary set of reliability-oriented metrics that directly reflect the
goals of the reliability layer:

| Metric | What it measures |
|--------|------------------|
| **Grounded answer rate** | Fraction of answers supported by retrieved evidence (groundedness ≥ 0.5) |
| **Unsupported claim rate** | Complement — answers the verifier cannot confirm |
| **Contradiction rate** | Answers that actively contradict the evidence (groundedness = 0) |
| **Abstention rate** | Fraction of queries the system refused to answer |
| **Correct abstention rate** | Abstained on queries that are truly unanswerable |
| **False abstention rate** | Abstained on queries that had a correct answer available |
| **Recovery attempt rate** | Fraction that triggered mechanism G |
| **Recovery success rate** | Among recovered queries: fraction that produced an answer |
| **Average trust score** | Mean H-score across answered queries |
| **Trust–correctness correlation** | Spearman ρ between trust score and ROUGE-L |

Gold labels for `is_answerable` are read from the extended benchmark:
- Standard questions (ids 1–25): `is_answerable=True`
- Extended `insufficient` and `adversarial` categories: `is_answerable=False`
- Extended `ambiguous`, `conflicting`, `multilingual`: `is_answerable=True` (hedged answer acceptable)

In [ ]:
from rag.evaluation.reliability_metrics import compute_reliability_metrics

# Gold labels for standard benchmark (all answerable)
gold_labels_standard = [
    {"query": q["question"],    "is_answerable": True, "gold_answer": q["answer"]}
    for q in qa_data
] + [
    {"query": q["question_de"], "is_answerable": True, "gold_answer": q["answer_de"]}
    for q in qa_data
]

# Gold labels for extended benchmark
UNANSWERABLE_CATEGORIES = {"insufficient", "adversarial"}
gold_labels_extended = [
    {
        "query":        q["question"],
        "is_answerable": q["category"] not in UNANSWERABLE_CATEGORIES,
        "gold_answer":  q.get("answer", ""),
    }
    for q in qa_extended
]

# Use standard labels for the standard benchmark run
rel_metrics = compute_reliability_metrics(reliable_results, gold_labels_standard)

rel_metrics_df = pd.DataFrame([rel_metrics]).T
rel_metrics_df.columns = ["value"]
print(rel_metrics_df.to_string())

rel_metrics_df.to_csv(RESULTS / "reliability_metrics_summary.csv")
print("Saved: reliability_metrics_summary.csv")

In [ ]:
# Abstention trigger breakdown
abstained_df = pd.DataFrame([r for r in reliable_results if r["abstained"]])
if len(abstained_df) > 0:
    trigger_counts = abstained_df["trigger"].value_counts()
    print("Abstention triggers:")
    print(trigger_counts.to_string())
    trigger_counts.to_csv(RESULTS / "abstention_triggers.csv")
    print("Saved: abstention_triggers.csv")
else:
    print("No abstentions in this run.")

## Section 6 — Ablation Study

An ablation study removes one component at a time to quantify its individual
contribution.  We test four system configurations:

| Condition | What changes | What this shows |
|-----------|-------------|------------------|
| **Full** | Default `ReliableOrchestrator` | Combined effect of all mechanisms |
| **No Recovery (G)** | `max_retries=0` | Impact of adaptive query rewriting / strategy switching |
| **No Groundedness (B+H)** | `groundedness_verifier=None, trust_scorer=None` | Impact of post-synthesis verification |
| **No Evidence Gate (A)** | Custom subclass that skips Mechanism A | Impact of pre-retrieval evidence screening |

We measure per condition: abstention rate, grounded answer rate, average ROUGE-L,
and average latency.  This reveals which mechanism contributes most to each
dimension of reliability.

**Note:** Each condition is run on the full standard benchmark (50 EN+DE queries).
The ablation run is the most time-consuming part of this notebook.

In [ ]:
class NoGateOrchestrator(ReliableOrchestrator):
    """Variant of ReliableOrchestrator that skips Mechanism A (evidence sufficiency gate).

    Mechanism A is the pre-synthesis quality gate.  Disabling it means the
    system always proceeds to answer synthesis regardless of evidence quality,
    mirroring the behaviour of the baseline.  Mechanisms B and H still run
    post-synthesis, but without A's abstention, their trust-based gate uses a
    simulated perfect evidence score (1.0) so only groundedness can trigger
    abstention.
    """

    def run(self, query, strategy="confidence", top_k=5):
        result = self._orchestrator.run(strategy=strategy, query=query, top_k=top_k)
        docs   = result["documents"]
        answer = self._orchestrator.synthesizer.generate(query, docs)
        trace  = ["[NoGate] Evidence gate skipped — always synthesize."]

        groundedness = self._run_grounder(query, answer, docs, trace)
        # Fake a perfect evidence report so trust gate works normally
        class _FakeReport:
            score = 1.0
            sufficient = True
        trust = self._run_trust(_FakeReport(), groundedness, trace)

        return ReliableResponse(
            query=query,
            strategy=strategy,
            answer=answer,
            abstained=False,
            evidence_report=None,
            documents=docs,
            trace=trace,
            groundedness_score=groundedness,
            trust_score=trust,
        )

In [ ]:
# Build the four ablation variants
conditions = {
    "Full (A+G+B+H+E)": reliable_orchestrator,
    "No Recovery (−G)": ReliableOrchestrator(
        orchestrator=baseline_orchestrator,
        embed_fn=embed_fn,
        max_retries=0,
        openai_client=openai_client,
        groundedness_verifier=groundedness_verifier,
        trust_scorer=trust_scorer,
    ),
    "No Groundedness (−B−H)": ReliableOrchestrator(
        orchestrator=baseline_orchestrator,
        embed_fn=embed_fn,
        max_retries=2,
        openai_client=openai_client,
        groundedness_verifier=None,
        trust_scorer=None,
    ),
    "No Evidence Gate (−A)": NoGateOrchestrator(
        orchestrator=baseline_orchestrator,
        embed_fn=embed_fn,
        max_retries=2,
        openai_client=openai_client,
        groundedness_verifier=groundedness_verifier,
        trust_scorer=trust_scorer,
    ),
}

# Build flat list of (question, gold_answer) for the standard benchmark
all_queries = []
for q in qa_data:
    all_queries.append((q["question"],    q["answer"]))
    all_queries.append((q["question_de"], q["answer_de"]))

print(f"Running ablation on {len(all_queries)} queries × {len(conditions)} conditions...")

In [ ]:
from rouge_score import rouge_scorer as _rs
_rouge = _rs.RougeScorer(["rougeL"], use_stemmer=False)

ablation_rows = []
for cond_name, orchestrator in conditions.items():
    records = []
    for query, gold in tqdm(all_queries, desc=cond_name, leave=False):
        t0 = time.time()
        r  = orchestrator.run(query=query, strategy="confidence", top_k=5)
        elapsed = time.time() - t0
        answer  = r.answer if not r.abstained else ""
        rouge_l = _rouge.score(gold.lower(), answer.lower())["rougeL"].fmeasure if answer else 0.0
        records.append({
            "abstained":          r.abstained,
            "groundedness_score": r.groundedness_score,
            "rouge_l":            rouge_l,
            "latency":            elapsed,
        })

    df = pd.DataFrame(records)
    answered = df[~df["abstained"]]
    ablation_rows.append({
        "Condition":          cond_name,
        "Abstention Rate":    round(df["abstained"].mean(), 3),
        "Grounded Rate":      round(
            (answered["groundedness_score"] >= 0.5).mean()
            if len(answered) else 0.0, 3
        ),
        "Avg ROUGE-L":        round(df["rouge_l"].mean(), 3),
        "Avg Latency (s)":    round(df["latency"].mean(), 2),
    })

ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))
ablation_df.to_csv(RESULTS / "ablation_study.csv", index=False)
print("Saved: ablation_study.csv")

## Section 7 — Extended Benchmark Evaluation

The extended benchmark adds 15 challenging queries designed to stress-test the
reliability mechanisms.  Unlike the standard benchmark (factoid ETH questions
with known answers), these cases require the system to:

- **Ambiguous**: ask for clarification rather than guessing
- **Insufficient evidence**: abstain when the corpus cannot support an answer
- **Conflicting evidence**: give a hedged answer noting the discrepancy
- **Adversarial**: detect and reject false premises
- **Multilingual / code-switched**: handle German or mixed EN/DE queries

We measure *correct behavior* per category: whether the system's actual routing
decision (answer / abstain / clarify) matches the `expected_behavior` field in the
benchmark JSON.

In [ ]:
print("Running reliable pipeline on extended benchmark...")
extended_results = []

for q in tqdm(qa_extended, desc="Extended benchmark"):
    t0  = time.time()
    r   = reliable_orchestrator.run(query=q["question"], strategy="confidence", top_k=5)
    elapsed = time.time() - t0

    if r.abstained:
        actual_behavior = "abstain" if r.abstention.trigger != "clarify" else "clarify"
        if r.abstention and r.abstention.trigger:
            actual_behavior = "clarify" if "clarif" in r.abstention.trigger else "abstain"
    else:
        actual_behavior = "answer"

    expected = q["expected_behavior"]
    # hedged counts as answer for now (system answered with some content)
    is_correct = (
        actual_behavior == expected
        or (expected == "hedged" and actual_behavior == "answer")
        or (expected == "clarify" and actual_behavior in ("abstain", "clarify"))
    )

    extended_results.append({
        "id":              q["id"],
        "category":        q["category"],
        "question":        q["question"],
        "expected":        expected,
        "actual":          actual_behavior,
        "correct":         is_correct,
        "abstained":       r.abstained,
        "trigger":         r.abstention.trigger if r.abstained and r.abstention else None,
        "answer_snippet":  r.answer[:120] if r.answer else "",
        "ev_score":        r.evidence_report.score if r.evidence_report else None,
        "trust_score":     r.trust_score,
        "latency":         round(elapsed, 2),
    })

ext_df = pd.DataFrame(extended_results)
ext_df.to_csv(RESULTS / "extended_benchmark_results.csv", index=False)
print("Saved: extended_benchmark_results.csv")

In [ ]:
# Per-category summary
category_summary = ext_df.groupby("category").agg(
    total=("id", "count"),
    correct=("correct", "sum"),
).assign(accuracy=lambda df: (df["correct"] / df["total"]).round(2))

print("Extended benchmark results per category:")
print(category_summary.to_string())

overall_correct = ext_df["correct"].sum()
print(f"\nOverall: {overall_correct}/{len(ext_df)} correct ({overall_correct/len(ext_df):.1%})")

category_summary.to_csv(RESULTS / "extended_benchmark_summary.csv")
print("Saved: extended_benchmark_summary.csv")

## Section 8 — Qualitative Examples

Quantitative metrics summarise system behavior in aggregate.  Qualitative examples
reveal *how* the system behaves in specific cases — where it succeeds gracefully,
where recovery saves it, and where it fails despite the safety mechanisms.

Five representative cases are selected from the benchmark runs:

1. **Successful grounded answer** — high trust score, answer matches gold
2. **Revised answer after critique** — Mechanism G rewrote the query; the second
   retrieval attempt produced a better evidence set and a correct answer
3. **Clarification case** — ambiguous query triggers abstention with useful guidance
4. **Abstention case** — evidence gate correctly withheld an unanswerable query
5. **Difficult failure case** — system answered confidently but ROUGE-L is low

In [ ]:
def print_example(title, record, include_trace=False):
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"  {title}")
    print(sep)
    print(f"  Query     : {record.get('query', '')}")
    print(f"  Abstained : {record.get('abstained', False)}")
    if record.get("abstained"):
        print(f"  Trigger   : {record.get('trigger', '')}")
    else:
        ans = str(record.get("answer") or record.get("answer_snippet") or "")[:200]
        print(f"  Answer    : {ans}")
    print(f"  Trust     : {record.get('trust_score', 'N/A')}")
    print(f"  EV Score  : {record.get('ev_score', 'N/A')}")
    print(f"  ROUGE-L   : {record.get('rouge_l', 'N/A')}")
    if include_trace and record.get("trace"):
        print("  Trace (last 5):")
        for entry in record["trace"][-5:]:
            print(f"    {entry}")

rel_df_full = pd.DataFrame(reliable_results)

# 1. Successful grounded answer — highest trust_score among answered queries
answered_mask = (~rel_df_full["abstained"]) & (rel_df_full["trust_score"].notna())
if answered_mask.any():
    best = rel_df_full[answered_mask].sort_values("trust_score", ascending=False).iloc[0]
    print_example("1. Successful Grounded Answer", best.to_dict())

# 2. Revised answer after critique — query was recovered (recoveries > 0) and answered
recovered_mask = (~rel_df_full["abstained"]) & (rel_df_full["recoveries"] > 0)
if recovered_mask.any():
    rec = rel_df_full[recovered_mask].iloc[0]
    print_example("2. Revised Answer After Recovery (Mechanism G)", rec.to_dict())
else:
    print("\n[No recovery+answer case found in standard benchmark — check extended benchmark.]")

# 3. Clarification case — from extended benchmark ambiguous category
clarify_cases = ext_df[ext_df["expected"] == "clarify"]
if len(clarify_cases) > 0:
    cl = clarify_cases.iloc[0]
    print_example("3. Clarification Case (Extended Benchmark)", cl.to_dict())

# 4. Abstention case — insufficient evidence category
insuf_cases = ext_df[ext_df["category"] == "insufficient"]
if len(insuf_cases) > 0:
    ab = insuf_cases.iloc[0]
    print_example("4. Correct Abstention (Insufficient Evidence)", ab.to_dict())

# 5. Failure case — answered but ROUGE-L is low (confident but wrong)
if "rouge_l" in rel_df_full.columns:
    failure_mask = (~rel_df_full["abstained"]) & (rel_df_full["rouge_l"].notna())
    if failure_mask.any():
        worst = rel_df_full[failure_mask].sort_values("rouge_l").iloc[0]
        print_example("5. Difficult Failure Case (Low ROUGE, System Answered)", worst.to_dict())

In [ ]:
# Save qualitative examples as structured JSON for report
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    answered_mask = (~rel_df_full["abstained"]) & (rel_df_full["trust_score"].notna())
    if answered_mask.any():
        best = rel_df_full[answered_mask].sort_values("trust_score", ascending=False).iloc[0]
        print_example("1. Successful Grounded Answer", best.to_dict())
    recovered_mask = (~rel_df_full["abstained"]) & (rel_df_full["recoveries"] > 0)
    if recovered_mask.any():
        rec = rel_df_full[recovered_mask].iloc[0]
        print_example("2. Revised Answer After Recovery (Mechanism G)", rec.to_dict())
    if len(clarify_cases) > 0:
        print_example("3. Clarification Case (Extended Benchmark)", clarify_cases.iloc[0].to_dict())
    if len(insuf_cases) > 0:
        print_example("4. Correct Abstention (Insufficient Evidence)", insuf_cases.iloc[0].to_dict())
    if "rouge_l" in rel_df_full.columns and failure_mask.any():
        worst = rel_df_full[failure_mask].sort_values("rouge_l").iloc[0]
        print_example("5. Difficult Failure Case (Low ROUGE, System Answered)", worst.to_dict())

with open(RESULTS / "qualitative_examples.txt", "w", encoding="utf-8") as f:
    f.write(buf.getvalue())
print("Saved: qualitative_examples.txt")

## Section 9 — Visualizations

All figures are saved to `RESULTS/figures/` as PNG files so they can be embedded
directly in the Quarto report without needing to re-run the notebook.

Figures produced:
- `ir_metrics_bar.png` — P@k, MRR, NDCG comparison (Baseline vs. Reliable)
- `answer_quality_bar.png` — BLEU, ROUGE-1, ROUGE-L, Semantic Similarity
- `reliability_metrics_bar.png` — reliability metric summary
- `ablation_heatmap.png` — 4-condition ablation table as a heatmap
- `abstention_triggers_pie.png` — abstention trigger breakdown
- `trust_vs_rouge_scatter.png` — trust score vs. ROUGE-L (calibration)

In [ ]:
# Helper: save and show a figure
def save_fig(name, fig=None):
    path = FIGURES / name
    (fig or plt).savefig(path, bbox_inches="tight", dpi=150)
    print(f"Saved: {path}")
    plt.show()
    plt.close("all")

In [ ]:
# Figure 1: IR metrics comparison
ir_df = pd.read_csv(RESULTS / "ir_metrics_comparison.csv")
key_metrics = [c for c in ir_df.columns
               if any(k in c for k in ("map", "ndcg", "recip", "recall", "P_"))]
key_metrics = key_metrics[:8]  # keep readable

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(key_metrics))
w = 0.35
for i, (_, row) in enumerate(ir_df.iterrows()):
    vals = [float(row[m]) for m in key_metrics]
    ax.bar(x + i * w, vals, w, label=row["Strategy"])
ax.set_xticks(x + w / 2)
ax.set_xticklabels([m.replace("_", "@").replace("cut", "") for m in key_metrics],
                   rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("IR Retrieval Metrics: Baseline vs. Reliable System")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
save_fig("ir_metrics_bar.png")

In [ ]:
# Figure 2: Answer quality comparison
aq_df = pd.read_csv(RESULTS / "answer_quality_comparison.csv", index_col=0)
plot_cols = [c for c in ["bleu", "rouge1", "rougeL", "semantic_similarity"] if c in aq_df.columns]

fig, ax = plt.subplots(figsize=(10, 5))
aq_df[plot_cols].plot(kind="bar", ax=ax, rot=20)
ax.set_ylabel("Score")
ax.set_title("Answer Quality: BLEU / ROUGE / Semantic Similarity")
ax.set_xlabel("")
ax.legend(loc="upper right")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
save_fig("answer_quality_bar.png")

In [ ]:
# Figure 3: Reliability metrics
rm_df = pd.read_csv(RESULTS / "reliability_metrics_summary.csv", index_col=0)
rm_df.columns = ["value"]
rm_plot = rm_df[rm_df["value"].apply(lambda v: str(v) != "nan")].copy()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["steelblue" if "rate" in idx else "coral" for idx in rm_plot.index]
ax.barh(rm_plot.index, rm_plot["value"].astype(float), color=colors)
ax.set_xlabel("Score / Rate")
ax.set_title("Reliability-Oriented Metrics Summary")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.5, label="0.5 reference")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
save_fig("reliability_metrics_bar.png")

In [ ]:
# Figure 4: Ablation heatmap
abl_df = pd.read_csv(RESULTS / "ablation_study.csv", index_col=0)
numeric_cols = [c for c in abl_df.columns if c != "Condition"]

fig, ax = plt.subplots(figsize=(10, 4))
data = abl_df[numeric_cols].astype(float)
sns.heatmap(
    data, annot=True, fmt=".3f", cmap="YlOrRd",
    ax=ax, linewidths=0.5,
    yticklabels=abl_df["Condition"] if "Condition" in abl_df.columns else abl_df.index,
)
ax.set_title("Ablation Study: Impact of Individual Mechanisms")
plt.tight_layout()
save_fig("ablation_heatmap.png")

In [ ]:
# Figure 5: Abstention triggers pie chart
if (RESULTS / "abstention_triggers.csv").exists():
    trig_df = pd.read_csv(RESULTS / "abstention_triggers.csv", index_col=0)
    trig_df.columns = ["count"]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(trig_df["count"], labels=trig_df.index, autopct="%1.0f%%", startangle=90)
    ax.set_title("Abstention Trigger Breakdown")
    plt.tight_layout()
    save_fig("abstention_triggers_pie.png")
else:
    print("No abstentions — pie chart skipped.")

In [ ]:
# Figure 6: Trust score vs. ROUGE-L scatter (confidence–correctness alignment)
plot_data = rel_df_full[
    (~rel_df_full["abstained"])
    & rel_df_full["trust_score"].notna()
    & rel_df_full.get("rouge_l", pd.Series([None] * len(rel_df_full))).notna()
].copy()

if len(plot_data) >= 3:
    from scipy.stats import spearmanr
    rho, pval = spearmanr(plot_data["trust_score"], plot_data["rouge_l"])

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(plot_data["trust_score"], plot_data["rouge_l"], alpha=0.7, edgecolors="k", s=60)
    ax.set_xlabel("Trust Score (H)")
    ax.set_ylabel("ROUGE-L vs. Gold Answer")
    ax.set_title(f"Trust Score vs. Answer Quality (Spearman ρ = {rho:.2f}, p = {pval:.3f})")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_fig("trust_vs_rouge_scatter.png")
    print(f"Spearman ρ = {rho:.3f}, p-value = {pval:.4f}")
else:
    print("Not enough data points for scatter plot.")

In [ ]:
print("=" * 60)
print("All evaluation outputs saved to Drive:")
for f in sorted(RESULTS.rglob("*.csv")) + sorted(RESULTS.rglob("*.json")) + sorted(FIGURES.rglob("*.png")):
    size = f.stat().st_size / 1024
    print(f"  {f.relative_to(ROOT)}  ({size:.1f} KB)")
print("=" * 60)
print("Download these files to report/data/ and report/images/ to render the Quarto report.")